# PowerNZ En Google Colab

Este notebook es la forma simple cuando Kaggle no quiere arrancar.

## Pasos

1. En Colab, voy a `Entorno de ejecucion` > `Cambiar tipo de entorno de ejecucion`.
2. En `Acelerador por hardware`, elijo `GPU` si esta disponible.
3. Ejecuto las celdas en orden.
4. Cuando me lo pida, subo mi video.
5. Descargo `powernz_analizado.mp4`.

## 1. Configuracion

Cambio solo `EXERCISE` si hace falta:

- `deadlift` = peso muerto
- `squat` = sentadilla
- `bench` = press banca

In [ ]:
EXERCISE = "deadlift"  # deadlift, squat o bench
PLATE_DIAMETER_PX = 120
OUTPUT_NAME = "powernz_analizado.mp4"

REPO_URL = "https://github.com/juanrdzmb/PowerNZ.git"
BRANCH = "codex/model-download-and-launcher"

print("Configuracion lista")

## 2. Instalar PowerNZ Y Descargar Modelos

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

workdir = Path("/content")
repo_dir = workdir / "PowerNZ"

if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)

subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["python", "model_downloader.py"], check=True)

print("PowerNZ y modelos listos")

## 3. Subir Video

Al ejecutar esta celda aparece un boton para elegir el video desde mi PC.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No subi ningun video")

video_name = next(iter(uploaded.keys()))
VIDEO_PATH = str(Path("/content/PowerNZ") / video_name)

print("Video listo:", VIDEO_PATH)

## 4. Analizar Video

In [ ]:
from pathlib import Path
import subprocess

output_path = Path("/content") / OUTPUT_NAME

command = [
    "python", "main.py",
    "--input", VIDEO_PATH,
    "--output", str(output_path),
    "--exercise", EXERCISE,
    "--pose-backend", "yolo",
    "--plate-diameter-px", str(PLATE_DIAMETER_PX),
    "--output-format", "portrait-720",
    "--no-mobile-conversion",
]

print("Analizando. Esto puede tardar...")
subprocess.run(command, check=True)
print("Listo:", output_path)

## 5. Descargar Resultado

In [ ]:
from google.colab import files

files.download(str(output_path))